# UTC RAG Threshold Calibration

This notebook evaluates retrieval against the public UTC Master Knowledge Base. It selects the lowest threshold that retains every supported evaluation case while rejecting every out-of-scope question.

The latest calibration selected `SIMILARITY_THRESHOLD=0.35`: Recall@2 100%, MRR 94.44%, rejection accuracy 100%. Re-run it whenever the PDF, chunking, embedding model, or ranking logic changes.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.rag import retrieve

KNOWN_CASES = {
    'bisa benerin pc nggk?': 'Laptop dan Komputer',
    'apakah UTC menerima servis HP?': 'Handphone dan HP',
    'printer saya error bisa diperbaiki?': 'Printer',
    'dimana saya bisa ke UTC?': 'Lokasi UTC',
    'jadwal pengambilan service': 'Jadwal Pengambilan Unit',
    'berapa harga service pasti?': 'Biaya dan Estimasi',
    'apakah harus tahu kerusakannya?': 'Harus Tahu Kerusakannya',
    'bisa instal game bajakan?': 'Perangkat Lunak',
    'status laptop saya sudah selesai belum?': 'Kabar Perbaikan',
}
UNKNOWN_CASES = ('siapa yang piket hari ini?', 'resep nasi goreng', 'berapa nomor rekening UTC?')
THRESHOLDS = tuple(round(value / 100, 2) for value in range(20, 51, 5))

In [2]:
known_matches = {question: retrieve(question, limit=2) for question in KNOWN_CASES}
unknown_matches = {question: retrieve(question, limit=2) for question in UNKNOWN_CASES}

for question, matches in known_matches.items():
    print(question)
    for rank, match in enumerate(matches, start=1):
        print(f'  {rank}. {match["title"]} | semantic={match["semantic_score"]:.3f} lexical={match["lexical_score"]:.3f} final={match["score"]:.3f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

bisa benerin pc nggk?
  1. BAB 2. LAYANAN UTC - 2.1 Laptop dan Komputer | semantic=0.375 lexical=1.000 final=0.725
  2. BAB 7. PERTANYAAN UMUM - 7.1 Apakah UTC Bisa Memperbaiki HP | semantic=0.290 lexical=0.500 final=0.465
apakah UTC menerima servis HP?
  1. BAB 2. LAYANAN UTC - 2.2 Handphone dan HP | semantic=0.482 lexical=0.750 final=0.744
  2. BAB 7. PERTANYAAN UMUM - 7.1 Apakah UTC Bisa Memperbaiki HP | semantic=0.325 lexical=0.750 final=0.587
printer saya error bisa diperbaiki?
  1. BAB 2. LAYANAN UTC - 2.3 Printer | semantic=0.551 lexical=0.333 final=0.668
  2. BAB 7. PERTANYAAN UMUM - 7.2 Apakah Harus Tahu Kerusakannya | semantic=0.385 lexical=0.000 final=0.385
dimana saya bisa ke UTC?
  1. BAB 7. PERTANYAAN UMUM - 7.4 Di Mana Lokasi UTC | semantic=0.311 lexical=0.333 final=0.428
  2. BAB 6. INFORMASI OPERASIONAL - 6.1 Lokasi UTC | semantic=0.276 lexical=0.333 final=0.392
jadwal pengambilan service
  1. BAB 6. INFORMASI OPERASIONAL - 6.2 Jadwal Pengambilan Unit | semantic=0.879 

In [3]:
def evaluate(threshold):
    recall = sum(
        any(expected in match['title'] and match['score'] >= threshold for match in known_matches[question])
        for question, expected in KNOWN_CASES.items()
    ) / len(KNOWN_CASES)
    mrr = sum(
        next((1 / rank for rank, match in enumerate(known_matches[question], start=1) if expected in match['title']), 0)
        for question, expected in KNOWN_CASES.items()
    ) / len(KNOWN_CASES)
    rejection = sum(
        not any(match['score'] >= threshold for match in unknown_matches[question])
        for question in UNKNOWN_CASES
    ) / len(UNKNOWN_CASES)
    return {'threshold': threshold, 'recall': recall, 'mrr': mrr, 'rejection': rejection}

results = [evaluate(threshold) for threshold in THRESHOLDS]
for result in results:
    print(f"threshold={result['threshold']:.2f} recall@2={result['recall']:.2%} mrr={result['mrr']:.2%} rejection={result['rejection']:.2%}")

threshold=0.20 recall@2=100.00% mrr=94.44% rejection=33.33%
threshold=0.25 recall@2=100.00% mrr=94.44% rejection=66.67%
threshold=0.30 recall@2=100.00% mrr=94.44% rejection=66.67%
threshold=0.35 recall@2=100.00% mrr=94.44% rejection=100.00%
threshold=0.40 recall@2=88.89% mrr=94.44% rejection=100.00%
threshold=0.45 recall@2=66.67% mrr=94.44% rejection=100.00%
threshold=0.50 recall@2=66.67% mrr=94.44% rejection=100.00%


In [4]:
selected = next((result for result in results if result['recall'] == 1 and result['rejection'] == 1), None)
if selected is None:
    raise RuntimeError('No tested threshold satisfies full recall and rejection; expand the candidate range or review the corpus.')

print(f"Selected configuration: SIMILARITY_THRESHOLD={selected['threshold']:.2f}")

Selected configuration: SIMILARITY_THRESHOLD=0.35


## Applying the Result

Copy the selected line into `.env` and restart the API. `app/rag.py` deliberately has no threshold fallback: startup fails if the calibrated value is missing or invalid.